# Capa Silver - Catálogo de Productos

In [1]:
from spark_session import *
from delta.tables import *

## SparkSession

In [2]:
spark = create_spark_session(
    "retail-dlh-silver",
    configs=[
        configure_delta_extensions,
        configure_delta_catalog,
        configure_s3
    ]
)

## Extracción

In [3]:
# - Obtiene dataframe de la capa bronze
df_product = spark.table("bronze.product")

df_product_cost = spark.table("bronze.product_cost")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Unión

In [4]:
# Realizar un inner join (dataframe, metodo join, Transformación)
joined_df = df_product.join(df_product_cost, df_product.PRODUCT_ID == df_product_cost.PRODUCT_CODE, "inner")
joined_df = joined_df.drop("PRODUCT_CODE")
joined_df = joined_df.withColumnRenamed("COST", "PRODUCT_COST")


## Carga

In [5]:
if DeltaTable.isDeltaTable(spark, "silver.products"):
  # - Obtiene tabla delta de la capa silver
  deltaTblSilProducts = DeltaTable.forName(spark, "silver.products")

  ## --Inserta nuevos regitros, actualiza registros con cambios
  deltaTblSilProducts.alias('SILV') \
    .merge(
      joined_df.alias('BRON'),
      condition='BRON.PRODUCT_ID = SILV.PRODUCT_ID'
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()
else:
  ## --Inserta registros por primera vez
  joined_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.products")

## Vacuum

In [6]:
# - Actualiza la configuración para que pueda establecer un período de retención cero.
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

#- Ejecutemos el comando de VACUUM 
#spark.sql(f"VACUUM '{vPathSilv}PRODUCTS.delta' RETAIN 0 HOURS").show(truncate=False)

## Cierre de SparkSession

In [7]:
spark.stop()